# Two numpy matrices, side-by-side, with locked zoom

Each cell of the matrix renders as a crisp "big pixel" in Bokeh. Pan/zoom one plot and the other follows. Hovering a pixel in one plot highlights the same pixel in the other and shows live values in the per-plot titles plus a combined readout strip.

In [21]:
import numpy as np
import panel as pn
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.layouts import gridplot, column
from bokeh.models import (
    LinearColorMapper, ColorBar, ColumnDataSource,
    HoverTool, CrosshairTool, Span, CustomJS, Div,
)
from bokeh.palettes import Viridis256

pn.extension()
output_notebook()

Loading BokehJS ...

## Hover-link helper

Layers of feedback when the mouse is over either plot:
1. A red dashed crosshair on **both** plots (shared `Span` overlays).
2. An outlined square on the **other** plot at the matching `(col, row)`.
3. A `HoverTool` tooltip showing `col, row, value`.
4. **Dynamic per-plot titles** appended with each plot's own value at the cursor.
5. **Combined readout strip** above the grid: `(col, row) → A=…, B=…`.

In [22]:
def _add_hover_link(p1, p2, w, h, r1, r2, src_a, src_b, base_titles, combined_div, base_div_text):
    """Wire bidirectional hover linking between two image plots of the same grid (w, h).

    Updates: highlight rect on the other plot, both figure titles, and a Div with a combined
    'A=..., B=...' readout. base_titles and base_div_text are shown when the cursor is outside.
    """
    # 1. Linked crosshair: same Span instances added to two CrosshairTools.
    vline = Span(dimension="height", line_dash="dashed", line_width=1, line_color="red")
    hline = Span(dimension="width", line_dash="dashed", line_width=1, line_color="red")
    p1.add_tools(CrosshairTool(overlay=[vline, hline]))
    p2.add_tools(CrosshairTool(overlay=[vline, hline]))

    # 2. Highlight rect on the OTHER plot at the hovered pixel.
    hl_from_1 = ColumnDataSource(data=dict(x=[], y=[]))  # hovered on p1, drawn on p2
    hl_from_2 = ColumnDataSource(data=dict(x=[], y=[]))  # hovered on p2, drawn on p1
    p2.rect(x="x", y="y", width=1, height=1, source=hl_from_1,
            line_color="red", line_width=2, fill_alpha=0.0)
    p1.rect(x="x", y="y", width=1, height=1, source=hl_from_2,
            line_color="red", line_width=2, fill_alpha=0.0)

    # 3+4+5. mousemove updates: highlight rect, both titles, combined Div.
    move_code = """
    const ix = Math.floor(cb_obj.x);
    const iy = Math.floor(cb_obj.y);
    const arrA = src_a.data.image[0];
    const arrB = src_b.data.image[0];
    const fmt = (v) => (Math.abs(v) >= 1000 || (v !== 0 && Math.abs(v) < 0.01))
        ? v.toExponential(3) : v.toFixed(3);
    if (ix < 0 || ix >= w || iy < 0 || iy >= h) {
        if (src_hl.data.x.length) src_hl.data = {x: [], y: []};
        t1.text = base1;
        t2.text = base2;
        div.text = base_div;
        return;
    }
    src_hl.data = {x: [ix + 0.5], y: [iy + 0.5]};
    const va = fmt(arrA[iy * w + ix]);
    const vb = fmt(arrB[iy * w + ix]);
    t1.text = base1 + ': ' + va;
    t2.text = base2 + ': ' + vb;
    div.text = '(' + ix + ', ' + iy + ') \u2192 ' + base1 + '=' + va + ', ' + base2 + '=' + vb;
    """
    leave_code = """
    src_hl.data = {x: [], y: []};
    t1.text = base1;
    t2.text = base2;
    div.text = base_div;
    """
    common_args = dict(
        src_a=src_a, src_b=src_b, w=w, h=h,
        t1=p1.title, t2=p2.title, div=combined_div,
        base1=base_titles[0], base2=base_titles[1], base_div=base_div_text,
    )
    p1.js_on_event("mousemove", CustomJS(args=dict(src_hl=hl_from_1, **common_args), code=move_code))
    p1.js_on_event("mouseleave", CustomJS(args=dict(src_hl=hl_from_1, **common_args), code=leave_code))
    p2.js_on_event("mousemove", CustomJS(args=dict(src_hl=hl_from_2, **common_args), code=move_code))
    p2.js_on_event("mouseleave", CustomJS(args=dict(src_hl=hl_from_2, **common_args), code=leave_code))

    # Tooltip — pin to the image renderer so the highlight rect doesn't trigger it.
    tooltips = [("col, row", "$x{0}, $y{0}"), ("value", "@image")]
    p1.add_tools(HoverTool(tooltips=tooltips, renderers=[r1]))
    p2.add_tools(HoverTool(tooltips=tooltips, renderers=[r2]))

## Static version

In [23]:
def linked_image_pair(a, b, titles=("A", "B"), palette=Viridis256,
                      plot_size=420, shared_color_scale=False, flip_y=True):
    a, b = np.asarray(a), np.asarray(b)
    ha, wa = a.shape
    hb, wb = b.shape
    assert (ha, wa) == (hb, wb), "hover linking assumes both arrays share a grid"

    if shared_color_scale:
        lo = float(min(a.min(), b.min()))
        hi = float(max(a.max(), b.max()))
        ma = mb = LinearColorMapper(palette=palette, low=lo, high=hi)
    else:
        ma = LinearColorMapper(palette=palette, low=float(a.min()), high=float(a.max()))
        mb = LinearColorMapper(palette=palette, low=float(b.min()), high=float(b.max()))

    src_a = ColumnDataSource(data=dict(image=[a]))
    src_b = ColumnDataSource(data=dict(image=[b]))

    common = dict(
        width=plot_size, height=plot_size,
        tools="pan,wheel_zoom,box_zoom,reset,save",
        active_scroll="wheel_zoom",
        match_aspect=True,
    )
    y_range = (ha, 0) if flip_y else (0, ha)

    p1 = figure(title=titles[0], x_range=(0, wa), y_range=y_range, **common)
    r1 = p1.image(image="image", x=0, y=0, dw=wa, dh=ha, source=src_a, color_mapper=ma)
    p1.add_layout(ColorBar(color_mapper=ma, width=8), "right")

    p2 = figure(title=titles[1], x_range=p1.x_range, y_range=p1.y_range, **common)
    r2 = p2.image(image="image", x=0, y=0, dw=wb, dh=hb, source=src_b, color_mapper=mb)
    p2.add_layout(ColorBar(color_mapper=mb, width=8), "right")

    base_div_text = "(hover any plot)"
    combined_div = Div(
        text=base_div_text,
        styles={"font-family": "monospace", "font-size": "13px", "padding": "4px 8px"},
    )
    _add_hover_link(p1, p2, wa, ha, r1, r2, src_a, src_b, titles, combined_div, base_div_text)

    show(column(combined_div, gridplot([[p1, p2]], merge_tools=True)))

In [24]:
rng = np.random.default_rng(0)
A = rng.normal(size=(48, 64))
B = np.roll(A, shift=3, axis=1) + 0.3 * rng.normal(size=A.shape)

linked_image_pair(A, B, titles=("A", "B (shifted+noisy)"))

## Interactive: `B = f(A, param)` driven by a slider

`A` is fixed; `B` is recomputed live as the slider moves. Zoom is still locked, hover linking + dynamic titles still work on every frame. When the slider moves, titles reset (the previously-displayed value is now stale).

In [ ]:
def linked_pair_with_slider(A, transform, slider, titles=("A", "B"),
                             palette=Viridis256, plot_size=420, flip_y=True,
                             shared_color_scale=False, autoscale_b=True):
    """A is fixed; B = transform(A, slider.value) and re-renders on slider change.

    autoscale_b: rescale B's color mapper to its current min/max each update.
    Set False (and shared_color_scale=True) when you want the colorbars to be
    directly comparable while the slider moves.
    """
    A = np.asarray(A)
    h, w = A.shape
    B0 = np.asarray(transform(A, slider.value))

    if shared_color_scale:
        lo = float(min(A.min(), B0.min()))
        hi = float(max(A.max(), B0.max()))
        ma = LinearColorMapper(palette=palette, low=lo, high=hi)
        mb = LinearColorMapper(palette=palette, low=lo, high=hi)
    else:
        ma = LinearColorMapper(palette=palette, low=float(A.min()), high=float(A.max()))
        mb = LinearColorMapper(palette=palette, low=float(B0.min()), high=float(B0.max()))

    src_a = ColumnDataSource(data=dict(image=[A]))
    src_b = ColumnDataSource(data=dict(image=[B0]))

    common = dict(
        width=plot_size, height=plot_size,
        tools="pan,wheel_zoom,box_zoom,reset,save",
        active_scroll="wheel_zoom",
        match_aspect=True,
    )
    y_range = (h, 0) if flip_y else (0, h)

    p1 = figure(title=titles[0], x_range=(0, w), y_range=y_range, **common)
    r1 = p1.image(image="image", x=0, y=0, dw=w, dh=h, source=src_a, color_mapper=ma)
    p1.add_layout(ColorBar(color_mapper=ma, width=8), "right")

    p2 = figure(title=titles[1], x_range=p1.x_range, y_range=p1.y_range, **common)
    r2 = p2.image(image="image", x=0, y=0, dw=w, dh=h, source=src_b, color_mapper=mb)
    p2.add_layout(ColorBar(color_mapper=mb, width=8), "right")

    base_div_text = "(hover any plot)"
    combined_div = Div(
        text=base_div_text,
        styles={"font-family": "monospace", "font-size": "13px", "padding": "4px 8px"},
    )
    _add_hover_link(p1, p2, w, h, r1, r2, src_a, src_b, titles, combined_div, base_div_text)
    layout = column(combined_div, gridplot([[p1, p2]], merge_tools=True))

    def _update(event):
        B = np.asarray(transform(A, event.new))
        src_b.data = dict(image=[B])
        if autoscale_b and not shared_color_scale:
            mb.low = float(B.min())
            mb.high = float(B.max())
        # Stale: the displayed value was for the OLD B. Reset until next mousemove.
        p1.title.text = titles[0]
        p2.title.text = titles[1]
        combined_div.text = base_div_text

    slider.param.watch(_update, "value")
    return pn.Column(slider, pn.pane.Bokeh(layout))

In [ ]:
def gauss_blur_fft(A, sigma):
    """Pure-numpy isotropic Gaussian blur via FFT. sigma is in pixels."""
    if sigma <= 0:
        return A.astype(float, copy=True)
    h, w = A.shape
    fy = np.fft.fftfreq(h)[:, None]
    fx = np.fft.fftfreq(w)[None, :]
    K = np.exp(-2 * (np.pi * sigma) ** 2 * (fx**2 + fy**2))
    return np.fft.ifft2(np.fft.fft2(A) * K).real

yy, xx = np.mgrid[0:96, 0:128]
A2 = (
    np.sin(xx / 6.0) * np.cos(yy / 5.0)
    + 0.6 * np.exp(-((xx - 40) ** 2 + (yy - 30) ** 2) / 80)
    + 0.4 * rng.normal(size=(96, 128))
)

sigma_slider = pn.widgets.FloatSlider(
    name="sigma (Gaussian blur, pixels)",
    start=0.0, end=8.0, step=0.1, value=0.0,
)

linked_pair_with_slider(A2, gauss_blur_fft, sigma_slider,
                        titles=("A", "blur(A, \u03c3)"))